# CROSS APPLY i OUTER APPLY — notatki referencyjne (SQL Server)

Przykłady na modelu: `dim_Klienci` (ID_Klienta, Nazwa), `fact_Sprzedaz` (ID_Klienta, DataSprzedazy, Kwota, ID_Produktu), `dim_Placowki` (ID_Placowki, Miasto).

## 1. Czym jest `APPLY` — intuicja, zanim zobaczysz składnię

`JOIN` łączy dwie tabele na podstawie warunku porównującego kolumny — sam warunek `ON` **nie może** odwoływać się do tabeli-wyniku jakiegoś podzapytania, które samo w sobie zależałoby od bieżącego wiersza lewej tabeli w dowolny sposób (poza prostym porównaniem kolumn).

**`APPLY` rozwiązuje dokładnie ten problem: pozwala, żeby prawa strona była wyrażeniem tabelarycznym (podzapytaniem albo funkcją table-valued), które jest wykonywane osobno, od nowa, dla KAŻDEGO wiersza lewej tabeli, z dostępem do jego kolumn.** To jest fundamentalna różnica względem `JOIN` — prawa strona `APPLY` "wie", w jakim wierszu lewej tabeli aktualnie się znajduje, i może to wykorzystać w dowolnie złożony sposób (nie tylko przez proste porównanie `=`).

**Dwa warianty:**
- **`CROSS APPLY`** — semantycznie jak `INNER JOIN`: jeśli podzapytanie po prawej nie zwróci żadnego wiersza dla danego wiersza lewej tabeli, ten wiersz lewej tabeli **znika** z wyniku.
- **`OUTER APPLY`** — semantycznie jak `LEFT JOIN`: jeśli podzapytanie po prawej nie zwróci nic, wiersz lewej tabeli **zostaje**, z `NULL`-ami po stronie prawej.

## 2. Klasyczne zastosowanie #1 — TOP N per grupa

To jest najczęstszy, praktyczny powód sięgania po `APPLY` — coś, czego zwykły `JOIN` nie potrafi zrobić w ogóle bez dodatkowych sztuczek.

**Zadanie: dla każdego klienta pokaż jego 3 najwyższe transakcje.**

```sql
SELECT k.ID_Klienta, k.Nazwa, TopTransakcje.DataSprzedazy, TopTransakcje.Kwota
FROM dim_Klienci k
CROSS APPLY (
    SELECT TOP 3 s.DataSprzedazy, s.Kwota
    FROM fact_Sprzedaz s
    WHERE s.ID_Klienta = k.ID_Klienta
    ORDER BY s.Kwota DESC
) AS TopTransakcje;
```

**Dlaczego to działa, a zwykły `JOIN` by nie zadziałał:** podzapytanie wewnątrz `CROSS APPLY` zawiera `WHERE s.ID_Klienta = k.ID_Klienta` — odwołuje się do `k`, czyli do **bieżącego wiersza lewej tabeli**. To jest niedozwolone w standardowym `JOIN ... ON`, gdzie warunek musi łączyć całe tabele naraz, nie "wykonywać się od nowa per wiersz z parametrem z zewnątrz". `APPLY` semantycznie działa jak pętla: "dla każdego klienta k, wykonaj to podzapytanie z podstawionym ID_Klienta, weź TOP 3 wynik, dołącz do wiersza k".

**Klienci bez żadnej transakcji znikną z wyniku** (bo `CROSS APPLY` = wewnętrzne dopasowanie). Jeśli chcesz ich zachować (z `NULL` w kolumnach transakcji):

```sql
SELECT k.ID_Klienta, k.Nazwa, TopTransakcje.DataSprzedazy, TopTransakcje.Kwota
FROM dim_Klienci k
OUTER APPLY (
    SELECT TOP 3 s.DataSprzedazy, s.Kwota
    FROM fact_Sprzedaz s
    WHERE s.ID_Klienta = k.ID_Klienta
    ORDER BY s.Kwota DESC
) AS TopTransakcje;
```

## 3. Porównanie z alternatywą — funkcja okna `ROW_NUMBER`

To jest ważne porównanie, bo "TOP N per grupa" da się też zrobić bez `APPLY`, przez funkcję okna:

```sql
WITH Ranked AS (
    SELECT
        s.ID_Klienta, s.DataSprzedazy, s.Kwota,
        ROW_NUMBER() OVER (PARTITION BY s.ID_Klienta ORDER BY s.Kwota DESC) AS rn
    FROM fact_Sprzedaz s
)
SELECT k.ID_Klienta, k.Nazwa, r.DataSprzedazy, r.Kwota
FROM dim_Klienci k
JOIN Ranked r ON r.ID_Klienta = k.ID_Klienta AND r.rn <= 3;
```

**Kiedy `ROW_NUMBER` bywa wydajniejszy:** `ROW_NUMBER` skanuje `fact_Sprzedaz` **raz**, sortując i numerując w jednym przebiegu. `CROSS APPLY` z `TOP N` wewnątrz w niektórych planach wykonania powoduje, że silnik wykonuje osobne wyszukiwanie/sortowanie **dla każdego wiersza lewej tabeli** — przy dużej liczbie klientów to potencjalnie N osobnych operacji sortowania zamiast jednej zbiorczej.

**Kiedy `CROSS APPLY` bywa czytelniejszy/lepszy mimo to:** gdy logika "top N" jest bardziej złożona niż prosty `ORDER BY` (np. wymaga dodatkowych warunków, `JOIN`ów wewnątrz podzapytania, wywołania funkcji table-valued) — wtedy zapisanie tego jako czytelne podzapytanie w `APPLY` bywa prostsze do utrzymania niż upychanie całej logiki w jedno wyrażenie `PARTITION BY`.

**Reguła praktyczna:** dla prostego "TOP N wg jednej kolumny sortowania" — zacznij od `ROW_NUMBER`, zmierz plan wykonania (`SET STATISTICS IO, TIME ON` albo Actual Execution Plan) obu wersji na realnych danych, zanim wybierzesz. Nie zakładaj z góry, że jedno jest szybsze — to zależy od indeksów i skali, dokładnie tak, jak z każdym porównaniem wydajnościowym, które robiliśmy w DAX.

## 4. Klasyczne zastosowanie #2 — wywołanie funkcji table-valued per wiersz

`APPLY` to jedyny sposób na połączenie tabeli z **funkcją tabelaryczną (TVF), której parametr pochodzi z bieżącego wiersza** lewej tabeli.

```sql
SELECT k.ID_Klienta, k.Nazwa, hist.Miesiac, hist.SumaSprzedazy
FROM dim_Klienci k
CROSS APPLY dbo.fn_HistoriaSprzedazyKlienta(k.ID_Klienta) AS hist;
```

`dbo.fn_HistoriaSprzedazyKlienta(@ID_Klienta)` to funkcja table-valued zwracająca historię sprzedaży dla podanego klienta — `CROSS APPLY` wywołuje ją **osobno dla każdego klienta**, podstawiając `k.ID_Klienta` jako parametr. Zwykły `JOIN` nie mógłby tego zrobić — funkcji nie da się "połączyć" przez `ON`, bo to nie jest tabela, tylko wywołanie z parametrem.

## 5. Klasyczne zastosowanie #3 — `STRING_SPLIT` per wiersz (rozbijanie wartości oddzielonych przecinkiem)

Częsty scenariusz przy danych kadrowych/konfiguracyjnych: kolumna zawiera kilka wartości oddzielonych przecinkiem (np. lista uprawnień, lista tagów), i chcesz je rozbić na osobne wiersze.

```sql
SELECT p.[Numer osobowy], p.[Nazwisko i imię], Uprawnienie.value AS PojedynczeUprawnienie
FROM dim_baza_kadrowa p
CROSS APPLY STRING_SPLIT(p.Uprawnienia, ',') AS Uprawnienie;
```

Dla pracownika z `Uprawnienia = 'Admin,Kierownik,Raporty'` dostaniesz 3 osobne wiersze — jeden na każde uprawnienie. `STRING_SPLIT` to funkcja table-valued (zwraca tabelę jednokolumnową `value`), więc **musi** być połączona przez `APPLY`, nie `JOIN` — bo jej wynik zależy od wartości `p.Uprawnienia` w bieżącym wierszu.

**Pracownik z `NULL` w `Uprawnienia`** zniknie przy `CROSS APPLY` (bo `STRING_SPLIT(NULL, ',')` nie zwraca wierszy) — użyj `OUTER APPLY`, jeśli chcesz go zachować z `PojedynczeUprawnienie = NULL`.

## 6. `OUTER APPLY` z agregacją — częsty, praktyczny wzorzec

**Zadanie: dla każdej placówki pokaż datę i kwotę jej ostatniej transakcji, nawet jeśli placówka jeszcze żadnej nie miała.**

```sql
SELECT pl.ID_Placowki, pl.Miasto, Ostatnia.DataSprzedazy, Ostatnia.Kwota
FROM dim_Placowki pl
OUTER APPLY (
    SELECT TOP 1 s.DataSprzedazy, s.Kwota
    FROM fact_Sprzedaz s
    WHERE s.ID_Placowki = pl.ID_Placowki
    ORDER BY s.DataSprzedazy DESC
) AS Ostatnia;
```

Placówki bez żadnej sprzedaży dostają `NULL` w `DataSprzedazy`/`Kwota`, ale **nie znikają z wyniku** — to jest dokładnie różnica względem `CROSS APPLY` z sekcji 2, tu świadomie zastosowana, bo chcemy pełną listę placówek niezależnie od tego, czy mają dane.

**Częsty błąd, na który warto uważać:** jeśli po tym `OUTER APPLY` dodasz `WHERE Ostatnia.Kwota > 1000`, **efektywnie zamienisz to z powrotem w coś jak `INNER JOIN`** — warunek w `WHERE` na kolumnie z prawej strony `OUTER APPLY` odfiltruje też te `NULL`-owe wiersze (bo `NULL > 1000` nigdy nie jest `TRUE`), niwecząc cel użycia `OUTER APPLY`. Jeśli chcesz zachować placówki bez sprzedaży mimo warunku na kwocie, warunek musi uwzględniać `NULL` jawnie: `WHERE Ostatnia.Kwota > 1000 OR Ostatnia.Kwota IS NULL`.

## 7. Podsumowanie — kiedy `APPLY`, kiedy zwykły `JOIN`

| Potrzebujesz | Rozwiązanie |
|---|---|
| Połączyć dwie tabele przez proste porównanie kolumn | `JOIN` — prostsze, lepiej zoptymalizowane, pierwszy wybór |
| Podzapytanie po prawej stronie musi odwoływać się do kolumn bieżącego wiersza lewej tabeli (poza prostym `=`) | `APPLY` (`CROSS`/`OUTER` zależnie, czy chcesz zachować wiersze bez dopasowania) |
| TOP N per grupa (prosty `ORDER BY`) | Zacznij od `ROW_NUMBER()` + `PARTITION BY`, porównaj z `CROSS APPLY` na realnych danych |
| TOP N per grupa (złożona logika wewnątrz) | `CROSS APPLY` z podzapytaniem — czytelniejsze niż komplikowanie jednego wyrażenia okna |
| Wywołanie funkcji table-valued z parametrem z bieżącego wiersza | `APPLY` — jedyna opcja, `JOIN` się nie nadaje |
| Rozbicie kolumny z wartościami rozdzielonymi separatorem na wiersze | `CROSS APPLY STRING_SPLIT(...)` (albo `OUTER APPLY`, jeśli chcesz zachować wiersze z `NULL`) |
| Chcesz zachować wiersze lewej tabeli bez dopasowania, ale filtrujesz po kolumnie z prawej | Uważaj na `WHERE` po `OUTER APPLY` — dodaj `OR kolumna IS NULL`, inaczej efekt jak `INNER JOIN` |